# PowerPlus — Data Processing & Feature Engineering

This notebook prepares the three PowerPlus data sources for electricity-demand forecasting:

1. **Minute-level electricity data** → daily electricity consumption in **kWh** (target).
2. **Household metadata** → household/appliance characteristics.
3. **City-wise weather data** → daily weather features.

The final output is a clean modeling dataset for the next milestone: **Decision Tree regression**.

Expected repository structure:

```text
Project-Electricity-Demand-Forecasting/
├── city-wise_house_dataset/
│   ├── Islamabad/
│   ├── Karachi/
│   ├── Lahore/
│   ├── Multan/
│   ├── Peshawar/
│   └── Skardu/
├── weather_dataset/
│   ├── Islamabad.csv
│   ├── Karachi.csv
│   ├── Lahore.csv
│   ├── Multan.csv
│   ├── Peshawar.csv
│   └── Skardu.csv
├── metadata_ultimate.xlsx
└── processed_data/
```

**OpenWeatherMap API is not required** for this notebook because historical weather CSV files are used.

In [1]:
# 1. IMPORT LIBRARIES

from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("Libraries loaded.")

Libraries loaded.


In [2]:
# 2. FIND PROJECT ROOT AND DEFINE PATHS

def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for p in candidates:
        if (p / "city-wise_house_dataset").exists():
            return p
    return Path.cwd()

PROJECT_ROOT = find_project_root()

ELECTRICITY_DIR = PROJECT_ROOT / "city-wise_house_dataset"
WEATHER_DIR = PROJECT_ROOT / "weather_dataset"
METADATA_FILE = PROJECT_ROOT / "metadata_ultimate.xlsx"
PROCESSED_DIR = PROJECT_ROOT / "processed_data"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ELECTRICITY_DIR:", ELECTRICITY_DIR)
print("WEATHER_DIR:", WEATHER_DIR)
print("METADATA_FILE:", METADATA_FILE)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_ROOT: d:\Project-Electricity-Demand-Forecasting
ELECTRICITY_DIR: d:\Project-Electricity-Demand-Forecasting\city-wise_house_dataset
WEATHER_DIR: d:\Project-Electricity-Demand-Forecasting\weather_dataset
METADATA_FILE: d:\Project-Electricity-Demand-Forecasting\metadata_ultimate.xlsx
PROCESSED_DIR: d:\Project-Electricity-Demand-Forecasting\processed_data


In [3]:
# 3. CHECK FILE STRUCTURE

print("\nCity folders:")
if ELECTRICITY_DIR.exists():
    for p in sorted(ELECTRICITY_DIR.iterdir()):
        if p.is_dir():
            print(" -", p.name)
else:
    print("ERROR: city-wise_house_dataset not found.")

print("\nWeather files:")
if WEATHER_DIR.exists():
    for p in sorted(WEATHER_DIR.iterdir()):
        if p.is_file():
            print(" -", p.name)
else:
    print("ERROR: weather_dataset not found.")

print("\nMetadata file exists:", METADATA_FILE.exists())


City folders:
 - Islamabad
 - Karachi
 - Lahore
 - Multan
 - Peshawar
 - Skardu

Weather files:
 - Islamabad.csv
 - Karachi.csv
 - Lahore.csv
 - Multan.csv
 - Peshawar.csv
 - Skardu.csv

Metadata file exists: True


In [4]:
# 4. HELPER FUNCTIONS

def normalize_col(col):
    return re.sub(r"[^a-z0-9]+", "_", str(col).strip().lower()).strip("_")

def find_column(df, candidates):
    normalized = {normalize_col(c): c for c in df.columns}
    for candidate in candidates:
        key = normalize_col(candidate)
        if key in normalized:
            return normalized[key]
    for candidate in candidates:
        key = normalize_col(candidate)
        for ncol, original in normalized.items():
            if key in ncol or ncol in key:
                return original
    return None

def read_csv_safely(path):
    for encoding in ["utf-8", "utf-8-sig", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=encoding)
        except Exception:
            pass
    raise ValueError(f"Could not read {path}")

def infer_house_from_filename(path):
    match = re.search(r"house\s*#?\s*(\d+)", path.stem, flags=re.I)
    if match:
        return f"House#{int(match.group(1))}"
    return path.stem

def infer_city_from_electricity_path(path):
    relative = path.relative_to(ELECTRICITY_DIR)
    return relative.parts[0] if len(relative.parts) >= 2 else None

print("Helper functions ready.")

Helper functions ready.


## Electricity target calculation

Your electricity files are minute-level and contain:

```text
datetime | Usage (kW)
```

If each row represents one minute and `Usage` is instantaneous power in kW:

**Daily energy (kWh) = sum of minute-level kW readings / 60**

The resulting target column will be:

```text
electricity_kwh
```

We also keep coverage information so incomplete days can be identified.

In [ ]:
# 5. LOAD ALL MINUTE-LEVEL ELECTRICITY FILES

electricity_frames = []
electricity_errors = []

csv_files = sorted(ELECTRICITY_DIR.rglob("*.csv"))

print("Electricity CSV files found:", len(csv_files))

for file_path in csv_files:
    try:
        df = read_csv_safely(file_path)

        datetime_col = find_column(
            df, ["datetime", "date_time", "timestamp", "date"]
        )
        usage_col = find_column(
            df, ["usage_kw", "usage (kw)", "usage", "power_kw", "power"]
        )

        if datetime_col is None or usage_col is None:
            electricity_errors.append({
                "file": str(file_path),
                "reason": "datetime or usage column not found",
                "columns": str(list(df.columns))
            })
            continue

        out = pd.DataFrame({
            "datetime": pd.to_datetime(df[datetime_col], errors="coerce"),
            "usage_kw": pd.to_numeric(df[usage_col], errors="coerce")
        })

        out["house"] = infer_house_from_filename(file_path)
        out["city"] = infer_city_from_electricity_path(file_path)
        out["source_file"] = file_path.name

        out = out.dropna(subset=["datetime", "usage_kw"])
        out = out.sort_values("datetime")

        electricity_frames.append(out)

    except Exception as e:
        electricity_errors.append({
            "file": str(file_path),
            "reason": str(e),
            "columns": ""
        })

if not electricity_frames:
    raise ValueError("No usable electricity files were loaded.")

electricity_minute = pd.concat(electricity_frames, ignore_index=True)

print("Minute-level rows:", len(electricity_minute))
display(electricity_minute.head())

Electricity CSV files found: 69


In [ ]:
# 6. CLEAN MINUTE-LEVEL ELECTRICITY

negative_count = (electricity_minute["usage_kw"] < 0).sum()

electricity_minute.loc[
    electricity_minute["usage_kw"] < 0, "usage_kw"
] = np.nan

electricity_minute = electricity_minute.dropna(subset=["usage_kw"])

duplicate_count = electricity_minute.duplicated(
    subset=["city", "house", "datetime"]
).sum()

electricity_minute = electricity_minute.drop_duplicates(
    subset=["city", "house", "datetime"],
    keep="first"
)

print("Negative readings removed:", negative_count)
print("Duplicate timestamps removed:", duplicate_count)
print("Clean minute rows:", len(electricity_minute))

In [ ]:
# 7. CONVERT MINUTE ELECTRICITY TO DAILY KWH

electricity_minute["date"] = electricity_minute["datetime"].dt.normalize()

daily_electricity = (
    electricity_minute
    .groupby(["city", "house", "date"], as_index=False)
    .agg(
        electricity_kwh=("usage_kw", lambda x: x.sum() / 60.0),
        readings=("usage_kw", "size"),
        avg_power_kw=("usage_kw", "mean"),
        max_power_kw=("usage_kw", "max"),
        min_power_kw=("usage_kw", "min"),
    )
)

daily_electricity["expected_readings"] = 1440

daily_electricity["coverage_pct"] = (
    daily_electricity["readings"]
    / daily_electricity["expected_readings"]
    * 100
)

daily_electricity["low_coverage_flag"] = (
    daily_electricity["coverage_pct"] < 90
)

daily_electricity = daily_electricity.sort_values(
    ["city", "house", "date"]
).reset_index(drop=True)

print("Daily electricity rows:", len(daily_electricity))
display(daily_electricity.head(20))

In [ ]:
# 8. SAVE CLEAN DAILY ELECTRICITY

daily_electricity_file = PROCESSED_DIR / "daily_electricity_clean.csv"
daily_electricity.to_csv(daily_electricity_file, index=False)

print("Saved:", daily_electricity_file)

## Household metadata

Metadata is household-level information. It will be joined using:

```text
city + house
```

Examples include residents, property area, floors, construction year and appliance counts.

In [ ]:
# 9. LOAD METADATA

if not METADATA_FILE.exists():
    raise FileNotFoundError(f"Metadata file not found: {METADATA_FILE}")

metadata = pd.read_excel(METADATA_FILE)

metadata.columns = [
    re.sub(r"\s+", " ", str(c)).strip()
    for c in metadata.columns
]

print("Metadata shape:", metadata.shape)
print(metadata.columns.tolist())
display(metadata.head())

In [ ]:
# 10. STANDARDIZE METADATA KEYS

metadata_house_col = find_column(
    metadata, ["House", "House Name", "House#"]
)
metadata_city_col = find_column(
    metadata, ["City"]
)

print("House column:", metadata_house_col)
print("City column:", metadata_city_col)

if metadata_house_col is None or metadata_city_col is None:
    raise ValueError("House or City column could not be detected.")

metadata_clean = metadata.copy()

metadata_clean["house"] = metadata_clean[metadata_house_col].astype(str).str.strip()
metadata_clean["city"] = metadata_clean[metadata_city_col].astype(str).str.strip()

def standardize_house(value):
    value = str(value).strip()
    m = re.search(r"house\s*#?\s*(\d+)", value, flags=re.I)
    return f"House#{int(m.group(1))}" if m else value

metadata_clean["house"] = metadata_clean["house"].apply(standardize_house)

metadata_clean = metadata_clean.replace(
    ["#REF!", "#DIV/0!", "#VALUE!", "#N/A", "N/A", "NA", ""],
    np.nan
)

metadata_clean = metadata_clean.drop_duplicates(
    subset=["city", "house"], keep="first"
)

display(metadata_clean[["city", "house"]].head(20))

## Weather aggregation

Weather is hourly, while the target is daily.

We convert weather to one row per:

```text
city + date
```

Examples:

- temperature → daily mean and maximum
- humidity → daily mean and maximum
- precipitation → daily sum
- wind speed → mean and maximum
- pressure → mean
- solar radiation → mean and maximum
- solar energy → sum
- UV index → maximum

In [ ]:
# 11. WEATHER AGGREGATION FUNCTION

def aggregate_weather_file(file_path):
    df = read_csv_safely(file_path)
    df = df.dropna(axis=1, how="all")

    datetime_col = find_column(
        df, ["datetime", "date_time", "timestamp", "date"]
    )

    if datetime_col is None:
        raise ValueError(f"No datetime column in {file_path.name}")

    df["datetime"] = pd.to_datetime(
        df[datetime_col], errors="coerce"
    )
    df = df.dropna(subset=["datetime"])
    df["date"] = df["datetime"].dt.normalize()

    candidates = {
        "temperature": ["temperature", "temp"],
        "humidity": ["humidity"],
        "dew": ["dew", "dew_point", "dewpoint"],
        "precipitation": ["precipitation", "precip", "rain"],
        "wind_speed": ["wind_speed", "windspeed", "wind speed"],
        "wind_direction": ["wind_direction", "winddirection", "wind direction"],
        "pressure": ["pressure"],
        "solar_radiation": ["solar_radiation", "solarradiation"],
        "solar_energy": ["solar_energy", "solarenergy"],
        "uv_index": ["uv_index", "uvindex", "uv"],
    }

    for standard_name, options in candidates.items():
        found = find_column(df, options)
        if found is not None:
            df[standard_name] = pd.to_numeric(
                df[found], errors="coerce"
            )

    agg = {}

    for col in [
        "temperature", "humidity", "dew",
        "wind_speed", "wind_direction",
        "pressure", "solar_radiation"
    ]:
        if col in df.columns:
            agg[f"{col}_mean"] = (col, "mean")

    for col in [
        "temperature", "humidity", "wind_speed",
        "solar_radiation", "uv_index"
    ]:
        if col in df.columns:
            agg[f"{col}_max"] = (col, "max")

    for col in ["precipitation", "solar_energy"]:
        if col in df.columns:
            agg[f"{col}_sum"] = (col, "sum")

    if not agg:
        raise ValueError(
            f"No recognized weather fields in {file_path.name}"
        )

    daily = df.groupby("date").agg(**agg).reset_index()
    daily["city"] = file_path.stem.strip()

    return daily

In [ ]:
# 12. LOAD ALL WEATHER FILES

weather_frames = []
weather_errors = []

weather_files = sorted(WEATHER_DIR.glob("*.csv"))

print("Weather CSV files found:", len(weather_files))

for file_path in weather_files:
    try:
        weather_frames.append(
            aggregate_weather_file(file_path)
        )
    except Exception as e:
        weather_errors.append({
            "file": str(file_path),
            "reason": str(e)
        })

if not weather_frames:
    raise ValueError("No usable weather CSV files were loaded.")

daily_weather = pd.concat(weather_frames, ignore_index=True)

daily_weather = daily_weather.sort_values(
    ["city", "date"]
).reset_index(drop=True)

print("Daily weather rows:", len(daily_weather))
display(daily_weather.head(20))

In [ ]:
# 13. SAVE DAILY WEATHER

daily_weather_file = PROCESSED_DIR / "daily_weather.csv"
daily_weather.to_csv(daily_weather_file, index=False)

print("Saved:", daily_weather_file)

# Merge the three datasets

### 1. Electricity + metadata

Join on:

```text
city + house
```

### 2. Add weather

Join on:

```text
city + date
```

This creates a household-day dataset containing:

**household characteristics + appliances + historical electricity + weather**

In [ ]:
# 14. MERGE ELECTRICITY WITH METADATA

electricity_keys = daily_electricity[["city", "house"]].drop_duplicates()
metadata_keys = metadata_clean[["city", "house"]].drop_duplicates()

missing_metadata = (
    electricity_keys
    .merge(metadata_keys, on=["city", "house"], how="left", indicator=True)
)

missing_metadata = missing_metadata[
    missing_metadata["_merge"] == "left_only"
].drop(columns="_merge")

print("Household-city combinations without metadata:")
display(missing_metadata)

merged = daily_electricity.merge(
    metadata_clean,
    on=["city", "house"],
    how="left",
    suffixes=("", "_metadata")
)

print("After electricity + metadata:", merged.shape)

In [ ]:
# 15. ADD DAILY WEATHER

merged = merged.merge(
    daily_weather,
    on=["city", "date"],
    how="left"
)

merged = merged.sort_values(
    ["city", "house", "date"]
).reset_index(drop=True)

print("After weather merge:", merged.shape)
display(merged.head())

In [ ]:
# 16. WEATHER MERGE QUALITY CHECK

weather_columns = [
    c for c in daily_weather.columns
    if c not in ["city", "date"]
]

if weather_columns:
    weather_missing = (
        merged[weather_columns]
        .isna()
        .mean()
        .mul(100)
        .sort_values(ascending=False)
        .to_frame("missing_percent")
    )
    display(weather_missing)

print("Cities:", merged["city"].nunique())
print("Houses:", merged["house"].nunique())
print("Date range:", merged["date"].min(), "to", merged["date"].max())

# Calendar feature engineering

Calendar features help the model learn repeating patterns:

- year
- month
- day
- day of week
- week of year
- day of year
- weekend
- season

These are predictors. The target remains:

```text
electricity_kwh
```

In [ ]:
# 17. CREATE CALENDAR FEATURES

merged["year"] = merged["date"].dt.year
merged["month"] = merged["date"].dt.month
merged["day"] = merged["date"].dt.day
merged["day_of_week"] = merged["date"].dt.dayofweek
merged["week_of_year"] = merged["date"].dt.isocalendar().week.astype(int)
merged["day_of_year"] = merged["date"].dt.dayofyear
merged["is_weekend"] = (merged["day_of_week"] >= 5).astype(int)

def season_from_month(month):
    if month in [12, 1, 2]:
        return "Winter"
    if month in [3, 4, 5]:
        return "Spring"
    if month in [6, 7, 8]:
        return "Summer"
    return "Autumn"

merged["season"] = merged["month"].apply(season_from_month)

display(
    merged[
        ["date", "year", "month", "day_of_week",
         "is_weekend", "season"]
    ].head()
)

# Lag features

Lag features tell the model about the household's previous electricity demand.

Examples:

```text
lag_1_day_kwh  = yesterday
lag_7_day_kwh  = same household 7 days ago
lag_30_day_kwh = same household 30 days ago
```

Rolling averages summarize recent historical demand.

We use `shift(1)` before rolling calculations so the current day's target is not accidentally used as an input feature.

In [ ]:
# 18. CREATE LAG FEATURES

merged = merged.sort_values(
    ["city", "house", "date"]
).reset_index(drop=True)

group_cols = ["city", "house"]

for lag in [1, 2, 3, 7, 14, 30]:
    merged[f"lag_{lag}_day_kwh"] = (
        merged.groupby(group_cols)["electricity_kwh"]
        .shift(lag)
    )

shifted = (
    merged.groupby(group_cols)["electricity_kwh"]
    .shift(1)
)

shifted_group = shifted.groupby(
    [merged["city"], merged["house"]]
)

for window in [3, 7, 14, 30]:
    merged[f"rolling_{window}_day_avg_kwh"] = (
        shifted_group
        .transform(lambda x, w=window:
                   x.rolling(w, min_periods=1).mean())
    )

display(
    merged[
        ["city", "house", "date", "electricity_kwh",
         "lag_1_day_kwh", "lag_7_day_kwh",
         "lag_30_day_kwh", "rolling_7_day_avg_kwh"]
    ].head(40)
)

# Household and appliance features

The metadata already contains household and appliance variables.

We convert numeric-looking appliance fields to numeric values and create:

```text
house_age
total_appliance_count
```

Text fields such as Owner/Rented, ceiling type and WAPDA connection type are kept as categorical features. They will be one-hot encoded in the Decision Tree notebook.

In [ ]:
# 19. CLEAN NUMERIC HOUSEHOLD/APPLIANCE FEATURES

numeric_candidates = [
    "No. of people (Temp+Perm)",
    "No. of Permanent residents",
    "No. of Children (0-13)",
    "No. of Adults (14-60)",
    "No. of Seniors (above 60)",
    "No. of temporary residents",
    "Property Area (Marla)",
    "Covered Area",
    "No of Floors",
    "Build year of house",
    "Average Ceiling Height ft",
    "No. of rooms",
    "Number of Washrooms",
    "Number of Stores",
    "Air Conditioners",
    "Air Coolers",
    "Refrigerators",
    "Washing Machines",
    "LED Bulbs",
    "Tube Lights",
    "Celling Fans",
    "Wall Fans",
    "Stand Fans",
    "Water Dispensers",
    "Water Pumps",
    "Electric Cooker",
    "Electric heaters",
    "Electric Irons",
    "Sewing Machine",
    "Microwave Ovens",
    "Geysers",
    "UPS",
    "Other Electronic Devices",
]

for col in numeric_candidates:
    if col in merged.columns:
        merged[col] = pd.to_numeric(
            merged[col], errors="coerce"
        )

if "Build year of house" in merged.columns:
    merged["house_age"] = (
        merged["year"] - merged["Build year of house"]
    ).clip(lower=0, upper=200)

appliance_candidates = [
    "Air Conditioners",
    "Air Coolers",
    "Refrigerators",
    "Washing Machines",
    "LED Bulbs",
    "Tube Lights",
    "Celling Fans",
    "Wall Fans",
    "Stand Fans",
    "Water Dispensers",
    "Water Pumps",
    "Electric Cooker",
    "Electric heaters",
    "Electric Irons",
    "Sewing Machine",
    "Microwave Ovens",
    "Geysers",
    "UPS",
    "Other Electronic Devices",
]

appliance_columns = [
    c for c in appliance_candidates if c in merged.columns
]

if appliance_columns:
    merged["total_appliance_count"] = (
        merged[appliance_columns]
        .fillna(0)
        .sum(axis=1)
    )

print("Appliance columns used:")
print(appliance_columns)

In [ ]:
# 20. BASIC CLEANING AND DUPLICATE CHECK

merged = merged.replace([np.inf, -np.inf], np.nan)

before = len(merged)

merged = merged.drop_duplicates(
    subset=["city", "house", "date"],
    keep="first"
)

print("Duplicate daily rows removed:", before - len(merged))

merged = merged.sort_values(
    ["city", "house", "date"]
).reset_index(drop=True)

In [ ]:
# 21. DATA QUALITY REPORT

quality_report = pd.DataFrame({
    "column": merged.columns,
    "dtype": merged.dtypes.astype(str).values,
    "missing_count": merged.isna().sum().values
})

quality_report["missing_percent"] = (
    quality_report["missing_count"]
    / len(merged)
    * 100
)

quality_report = quality_report.sort_values(
    "missing_percent",
    ascending=False
)

display(quality_report.head(60))

# Model-ready dataset

Target:

```text
electricity_kwh
```

The dataset now contains:

### Historical electricity
- lag 1, 2, 3, 7, 14, 30 days
- rolling 3, 7, 14, 30 day averages

### Weather
- temperature
- humidity
- dew
- precipitation
- wind
- pressure
- solar radiation
- solar energy
- UV index

### Household
- residents
- property information
- construction year
- rooms
- appliances and electrical equipment

### Calendar
- month
- day of week
- weekend
- season

In [ ]:
# 22. SAVE MASTER MERGED DATASET

master_file = PROCESSED_DIR / "powerplus_daily_merged.csv"

merged.to_csv(
    master_file,
    index=False
)

print("Saved master dataset:", master_file)
print("Shape:", merged.shape)

In [ ]:
# 23. CREATE MODEL DATASET

required_history = [
    "lag_1_day_kwh",
    "lag_7_day_kwh",
    "lag_30_day_kwh"
]

model_data = merged.dropna(
    subset=["electricity_kwh"] + required_history
).copy()

model_file = PROCESSED_DIR / "powerplus_model_data.csv"

model_data.to_csv(
    model_file,
    index=False
)

print("Saved model dataset:", model_file)
print("Shape:", model_data.shape)

In [ ]:
# 24. FINAL TARGET CHECK

TARGET = "electricity_kwh"

print("TARGET:", TARGET)
print("\nTarget statistics:")
display(model_data[TARGET].describe().to_frame())

print("\nModel data preview:")
display(model_data.head())

In [ ]:
# 25. CITY SUMMARY

city_summary = (
    model_data
    .groupby("city")
    .agg(
        houses=("house", "nunique"),
        days=("date", "nunique"),
        records=("electricity_kwh", "size"),
        avg_daily_kwh=("electricity_kwh", "mean"),
        min_daily_kwh=("electricity_kwh", "min"),
        max_daily_kwh=("electricity_kwh", "max")
    )
    .reset_index()
)

display(city_summary)

In [ ]:
# 26. CREATE FEATURE LIST FOR NEXT NOTEBOOK

numeric_features = [
    c for c in [
        # Weather
        "temperature_mean", "temperature_max",
        "humidity_mean", "humidity_max",
        "dew_mean", "precipitation_sum",
        "wind_speed_mean", "wind_speed_max",
        "pressure_mean",
        "solar_radiation_mean", "solar_radiation_max",
        "solar_energy_sum", "uv_index_max",

        # Lag features
        "lag_1_day_kwh", "lag_2_day_kwh", "lag_3_day_kwh",
        "lag_7_day_kwh", "lag_14_day_kwh", "lag_30_day_kwh",
        "rolling_3_day_avg_kwh", "rolling_7_day_avg_kwh",
        "rolling_14_day_avg_kwh", "rolling_30_day_avg_kwh",

        # Calendar
        "year", "month", "day", "day_of_week",
        "week_of_year", "day_of_year", "is_weekend",

        # Household/appliances
        "No. of people (Temp+Perm)",
        "No. of Permanent residents",
        "No. of Children (0-13)",
        "No. of Adults (14-60)",
        "No. of Seniors (above 60)",
        "No. of temporary residents",
        "Property Area (Marla)",
        "Covered Area",
        "No of Floors",
        "Average Ceiling Height ft",
        "No. of rooms",
        "Number of Washrooms",
        "Number of Stores",
        "Air Conditioners", "Air Coolers",
        "Refrigerators", "Washing Machines",
        "LED Bulbs", "Tube Lights",
        "Celling Fans", "Wall Fans", "Stand Fans",
        "Water Dispensers", "Water Pumps",
        "Electric Cooker", "Electric heaters",
        "Electric Irons", "Sewing Machine",
        "Microwave Ovens", "Geysers", "UPS",
        "Other Electronic Devices",
        "house_age", "total_appliance_count"
    ]
    if c in model_data.columns
]

categorical_features = [
    c for c in [
        "city", "house", "Owner/Rented",
        "Wapda Connection type", "Ceiling Type",
        "Roof Type", "Flooring Type",
        "Interior Wall", "Exterior Wall",
        "Room Dimensions", "Kitchen", "Doors Type",
        "season", "Floor of Residency"
    ]
    if c in model_data.columns
]

feature_list = pd.DataFrame({
    "feature": numeric_features + categorical_features,
    "type": (
        ["numeric"] * len(numeric_features) +
        ["categorical"] * len(categorical_features)
    )
})

feature_file = PROCESSED_DIR / "powerplus_feature_list.csv"
feature_list.to_csv(feature_file, index=False)

display(feature_list)
print("Saved:", feature_file)

# 27. PROJECT MILESTONE STATUS

After this notebook runs successfully:

### Completed
- [x] Collect/import household metadata
- [x] Import minute-level electricity data
- [x] Clean electricity timestamps and values
- [x] Convert minute-level kW to daily kWh target
- [x] Import historical weather CSVs
- [x] Aggregate hourly weather to daily city-level features
- [x] Merge electricity + metadata
- [x] Merge weather + household-day records
- [x] Create calendar features
- [x] Create lag features
- [x] Create rolling historical demand features
- [x] Save master modeling dataset

### Next
- [ ] Train/test split by date
- [ ] Encode categorical variables
- [ ] Train Decision Tree Regressor
- [ ] Evaluate MAE, RMSE and R²
- [ ] Create 7-day forecast
- [ ] Create 30-day forecast
- [ ] Add future weather data/forecast if weather is used for future prediction
- [ ] Build consumer input/interface
- [ ] Present predicted electricity demand to the consumer